# SOLUTION: Lab 2 — MNIST Architecture Evolution
## MLP → CNN → CNN+LSTM → MiniViT

**Author:** Dr. Papa-Séga WADE | AIMS Applied GAAI - Module 0
---

In [1]:
# Optional: run this cell only if a package is missing.
"""
This cell installs the Python packages needed for the notebook.
Run it only if the current environment does not already have these packages.
In Google Colab, %pip installs packages inside the active notebook kernel.
"""
%pip install -q plotly pandas torchvision scikit-learn ipywidgets nbformat


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
"""Main PyTorch package: tensors, GPU support, and training utilities."""
import torch.nn as nn
"""torch.nn contains neural network layers such as Linear, Conv2d, LSTM, and LayerNorm."""
import torch.nn.functional as F
"""torch.nn.functional contains stateless functions such as relu and interpolation."""
import numpy as np
"""NumPy is used for numerical arrays outside PyTorch, especially before plotting."""
import plotly.express as px
"""Plotly Express gives quick high-level plotting functions, such as px.scatter and px.line."""
import plotly.graph_objects as go
"""Graph Objects gives lower-level control, useful for Heatmap traces."""
from plotly.subplots import make_subplots
"""make_subplots lets us create grids of images and attention maps."""
import pandas as pd
"""Pandas stores tabular results before plotting comparison dashboards."""
from torchvision import datasets, transforms
"""torchvision gives access to MNIST and image preprocessing transforms."""
from torch.utils.data import DataLoader
"""DataLoader creates mini-batches and shuffles training examples."""
from sklearn.manifold import TSNE
"""t-SNE projects high-dimensional embeddings into 2D for visualization."""

torch.manual_seed(42)
"""Fix PyTorch randomness to make results more comparable across runs."""
np.random.seed(42)
"""Fix NumPy randomness for reproducible visualizations and sampling."""
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
"""Use GPU if available, otherwise fall back to CPU."""
print(f'Device: {DEVICE}')
"""Display the device so students know where the model will run."""


Device: cpu


'Display the device so students know where the model will run.'

In [3]:
"""
MNIST preprocessing pipeline.

- transforms.ToTensor() converts each PIL image to a tensor with shape (1,28,28)
  and pixel values in [0,1].
- transforms.Normalize((0.1307,), (0.3081,)) standardizes MNIST using the
  dataset mean and standard deviation. This usually helps optimization.
- transforms.Compose(...) chains these two preprocessing steps and applies them
  to every image when it is loaded.
"""
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
"""Training split: download MNIST into ./data if it is not already present."""
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)
"""Test split: used only for evaluation, not for gradient updates."""

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
"""Training batches: shuffle=True changes the image order every epoch."""
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)
"""Test batches: shuffle=False keeps evaluation deterministic and easier to inspect."""

# --- Visualize MNIST: 4 rows x 8 cols heatmap grid ---
imgs, labels = next(iter(train_loader))
"""Take one mini-batch so we can display sample images before training."""
rows, cols = 4, 8
"""We display 32 images arranged as a 4-by-8 grid."""
fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[f"Label: {labels[i*cols+j].item()}" for i in range(rows) for j in range(cols)],
    vertical_spacing=0.06,
    horizontal_spacing=0.02
)
"""Create a Plotly subplot grid, with one subplot per digit image."""

for i in range(rows):
    """Loop over grid rows."""
    for j in range(cols):
        """Loop over grid columns."""
        idx = i * cols + j
        """Convert 2D grid position into the image index inside the batch."""
        img = imgs[idx].squeeze().numpy()
        """Remove the channel dimension: (1,28,28) -> (28,28), then convert to NumPy."""
        fig.add_trace(
            go.Heatmap(
                z=img[::-1],
                colorscale="Greys",
                showscale=False,
                hovertemplate="row=%{y}<br>col=%{x}<br>val=%{z:.2f}<extra></extra>"
            ),
            row=i+1,
            col=j+1
        )
        """Add one digit image as a heatmap in the correct subplot position."""

fig.update_layout(
    title=dict(text="MNIST Samples — Heatmap Grid", font=dict(size=18)),
    height=650,
    width=1000,
    plot_bgcolor="white",
    paper_bgcolor="white"
)
"""Set figure title, size, and background colors."""
fig.update_xaxes(showticklabels=False, showgrid=False)
"""Hide x-axis ticks and grid lines because pixel coordinates are not the focus."""
fig.update_yaxes(showticklabels=False, showgrid=False)
"""Hide y-axis ticks and grid lines for a cleaner image grid."""
fig.show()
"""Render the Plotly figure in the notebook."""

print(f"Dataset: {len(train_data):,} train / {len(test_data):,} test | image size: 28x28 | classes: 0-9")
"""Print a quick dataset summary for sanity checking."""


Dataset: 60,000 train / 10,000 test | image size: 28x28 | classes: 0-9


'Print a quick dataset summary for sanity checking.'

In [4]:
# ===== SHARED UTILITIES =====
def count_params(model):
    """
    Count the number of trainable parameters.

    We count only parameters with requires_grad=True because those are updated
    during training. This lets us compare CNN, CNN+LSTM, and MiniViT fairly.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(model, loader, epochs=5, lr=1e-3):
    """
    Train a model for a few epochs and return the test accuracy after each epoch.

    model: architecture to train.
    loader: training DataLoader.
    epochs: number of passes through the full training set.
    lr: learning rate for Adam.
    """
    model = model.to(DEVICE)
    """Move model weights to GPU if available, otherwise CPU."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    """Adam updates parameters using gradients computed by backpropagation."""
    criterion = nn.CrossEntropyLoss()
    """CrossEntropyLoss is standard for multi-class classification with logits."""
    history = []
    """Store test accuracy after each epoch for the final comparison plot."""
    for ep in range(epochs):
        """One epoch means one full pass over the training DataLoader."""
        model.train()
        """Training mode activates training-specific layers such as Dropout."""
        total_loss = 0
        """Accumulate loss values to print the average loss for the epoch."""
        for x, y in loader:
            """Loop over mini-batches of images x and labels y."""
            x, y = x.to(DEVICE), y.to(DEVICE)
            """Move each mini-batch to the same device as the model."""
            optimizer.zero_grad()
            """Reset gradients before computing the next backward pass."""
            loss = criterion(model(x), y)
            """Forward pass: model(x) gives logits, criterion compares logits to labels."""
            loss.backward()
            """Backward pass: compute gradients for all trainable parameters."""
            optimizer.step()
            """Optimizer step: update parameters using the gradients."""
            total_loss += loss.item()
            """Store the scalar loss value for logging."""
        acc = evaluate(model, test_loader)
        """Evaluate on test data after the epoch."""
        history.append(acc)
        """Append accuracy so we can plot learning curves later."""
        print(f'  Epoch {ep+1}/{epochs} | Loss: {total_loss/len(loader):.3f} | Test Acc: {acc:.2f}%')
    return history


def evaluate(model, loader):
    """Compute classification accuracy without updating model weights."""
    model.eval()
    """Evaluation mode disables training-specific behavior such as Dropout."""
    correct, total = 0, 0
    """Counters for correct predictions and total examples."""
    with torch.no_grad():
        """Disable gradient tracking to make evaluation faster and lighter."""
        for x, y in loader:
            """Loop over evaluation batches."""
            x, y = x.to(DEVICE), y.to(DEVICE)
            """Move data to the active device."""
            preds = model(x).argmax(dim=1)
            """Take the class index with the largest logit for each image."""
            correct += (preds == y).sum().item()
            """Count how many predictions match the labels."""
            total += y.size(0)
            """Add the batch size to the total number of evaluated examples."""
    return 100 * correct / total


def extract_embeddings(model, loader, n_batches=15):
    """
    Collect internal model representations for visualization.

    Each model implements get_embeddings(x). For MLP/CNN/CNN+LSTM, this is the
    penultimate representation. For MiniViT, it is the final CLS embedding.
    """
    model.eval()
    """Use evaluation mode so embeddings are stable."""
    all_embs, all_labels = [], []
    """Lists that will store embeddings and labels from several batches."""
    with torch.no_grad():
        """No gradients are needed when extracting embeddings."""
        for i, (x, y) in enumerate(loader):
            """Read batches from the loader and keep only the first n_batches."""
            if i >= n_batches:
                """Stop early to keep t-SNE fast enough for class time."""
                break
            emb = model.get_embeddings(x.to(DEVICE)).cpu().numpy()
            """Move images to device, compute embeddings, then move result back to CPU NumPy."""
            all_embs.append(emb)
            """Store this batch of embeddings."""
            all_labels.append(y.numpy())
            """Store labels as NumPy arrays for plotting."""
    return np.concatenate(all_embs), np.concatenate(all_labels)


def plot_tsne(embeddings, labels, title):
    """
    Project embeddings into 2D and plot them with one color per digit.

    n_components=2:
        We want a 2D plot, so each image embedding becomes one point (x, y).

    perplexity=30:
        Roughly controls how many neighbors each point tries to preserve.
        For a few thousand MNIST embeddings, 30 is a standard practical value.

    random_state=42:
        Fixes the random initialization so the plot is more reproducible.

    max_iter=800:
        Number of optimization iterations. More iterations can improve the layout
        but take longer to run.
    """
    print(f'  Running t-SNE: {title}...')
    """Display progress because t-SNE can take some time."""
    proj = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=800).fit_transform(embeddings)
    """Compute 2D coordinates from high-dimensional embeddings."""
    df = pd.DataFrame({'x': proj[:, 0], 'y': proj[:, 1], 'digit': labels.astype(str)})
    """Create a table with x/y coordinates and digit labels for Plotly."""
    fig = px.scatter(
        df,
        x='x',
        y='y',
        color='digit',
        title=f't-SNE — {title}',
        color_discrete_sequence=px.colors.qualitative.Plotly,
        width=720,
        height=560
    )
    """Build a colored scatter plot where each point is one MNIST image."""
    fig.update_traces(marker=dict(size=4, opacity=0.8))
    """Use small semi-transparent markers so dense regions remain readable."""
    fig.update_layout(
        xaxis=dict(showticklabels=False, title=''),
        yaxis=dict(showticklabels=False, title=''),
        legend_title='Digit'
    )
    """Hide numerical axes because t-SNE coordinates are relative, not physical units."""
    fig.show()
    """Render the plot in the notebook."""


histories = {}
"""Dictionary: model name -> list of test accuracies over epochs."""
results = {}
"""Dictionary: model name -> final test accuracy."""


'Dictionary: model name -> final test accuracy.'

---
## SOLUTION — Part 1: MLP

In [5]:
class MLP_MNIST(nn.Module):
    """
    A simple Multi-Layer Perceptron baseline.

    It ignores image geometry by flattening 28x28 pixels into one vector of 784 values.
    This is useful as a baseline because it has almost no spatial inductive bias.
    """
    def __init__(self, hidden=256):
        """Create the MLP layers."""
        super().__init__()
        """Initialize nn.Module internals."""
        self.fc1 = nn.Linear(784, hidden)
        """First fully connected layer: 784 pixel values -> hidden features."""
        self.fc2 = nn.Linear(hidden, hidden)
        """Second fully connected layer: hidden features -> richer hidden features."""
        self.head = nn.Linear(hidden, 10)
        """Classification head: hidden features -> 10 digit logits."""
        self.act = nn.ReLU()
        """ReLU introduces non-linearity after each hidden linear layer."""
        self.drop = nn.Dropout(0.2)
        """Dropout randomly drops hidden units during training to reduce overfitting."""

    def get_embeddings(self, x):
        """Return the representation just before classification."""
        x = x.view(x.size(0), -1)
        """Flatten image batch: (B,1,28,28) -> (B,784)."""
        x = self.act(self.fc1(x))
        """First hidden transformation: (B,784) -> (B,hidden)."""
        x = self.act(self.fc2(x))
        """Second hidden transformation: (B,hidden) -> (B,hidden)."""
        return x

    def forward(self, x):
        """Return logits for the 10 digit classes."""
        return self.head(self.drop(self.get_embeddings(x)))


print('=== MLP ===')
"""Print a section header for readability."""
mlp = MLP_MNIST()
"""Create the MLP model."""
print(f'Parameters: {count_params(mlp):,}')
"""Show how many trainable parameters the MLP has."""
histories['MLP'] = train_model(mlp, train_loader)
"""Train MLP and store accuracy after each epoch."""
results['MLP'] = histories['MLP'][-1]
"""Store final test accuracy for the dashboard."""


=== MLP ===
Parameters: 269,322
  Epoch 1/5 | Loss: 0.269 | Test Acc: 96.26%
  Epoch 2/5 | Loss: 0.106 | Test Acc: 97.27%
  Epoch 3/5 | Loss: 0.071 | Test Acc: 97.60%
  Epoch 4/5 | Loss: 0.054 | Test Acc: 97.50%
  Epoch 5/5 | Loss: 0.043 | Test Acc: 97.79%


'Store final test accuracy for the dashboard.'

In [6]:
embs, labs = extract_embeddings(mlp, test_loader)
"""Extract MLP hidden representations and labels from the test set."""
plot_tsne(embs, labs, 'MLP (256-dim)')
"""Plot the MLP embedding space in 2D using t-SNE."""


  Running t-SNE: MLP (256-dim)...


'Plot the MLP embedding space in 2D using t-SNE.'

---
## SOLUTION — Part 2: CNN

In [7]:
class CNN_MNIST(nn.Module):
    """
    Convolutional Neural Network baseline.

    Unlike the MLP, the CNN keeps spatial structure and learns local patterns
    such as strokes, corners, and small digit parts.
    """
    def __init__(self):
        """Create convolutional feature extractor and classifier layers."""
        super().__init__()
        """Initialize nn.Module internals."""
        """
        Convolutional feature extractor:
        - Conv2d(1,32,3,padding=1) keeps image size 28x28 and creates 32 feature maps.
        - MaxPool2d(2) divides height and width by 2: 28x28 -> 14x14.
        - Conv2d(32,64,3,padding=1) creates 64 feature maps while keeping 14x14.
        - A second MaxPool2d(2) gives 7x7 feature maps.
        """
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # (B,1,28,28) -> (B,32,28,28)
            nn.ReLU(),                                  # non-linearity after first convolution
            nn.MaxPool2d(2),                            # (B,32,28,28) -> (B,32,14,14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),# (B,32,14,14) -> (B,64,14,14)
            nn.ReLU(),                                  # non-linearity after second convolution
            nn.MaxPool2d(2),                            # (B,64,14,14) -> (B,64,7,7)
        )
        """Feature extractor: converts images into 64 feature maps of size 7x7."""
        self.fc1 = nn.Linear(64*7*7, 128)
        """64*7*7 flattened CNN features -> 128-dimensional embedding."""
        self.head = nn.Linear(128, 10)
        """128-dimensional embedding -> 10 digit logits."""
        self.drop = nn.Dropout(0.3)
        """Dropout regularizes the classifier part of the CNN."""

    def get_embeddings(self, x):
        """Return CNN representation before the final classification layer."""
        x = self.conv(x)
        """Apply convolutional feature extractor: expected shape (B,64,7,7)."""
        x = x.view(x.size(0), -1)
        """Flatten CNN feature maps: (B,64,7,7) -> (B,64*7*7)."""
        return F.relu(self.fc1(x))

    def forward(self, x):
        """Return logits for the 10 digit classes."""
        return self.head(self.drop(self.get_embeddings(x)))


print('=== CNN ===')
"""Print a section header for readability."""
cnn = CNN_MNIST()
"""Create the CNN model."""
print(f'Parameters: {count_params(cnn):,}')
"""Show CNN parameter count; target reference is 421,642 parameters."""
histories['CNN'] = train_model(cnn, train_loader)
"""Train CNN and store accuracy after each epoch."""
results['CNN'] = histories['CNN'][-1]
"""Store final CNN test accuracy."""


=== CNN ===
Parameters: 421,642
  Epoch 1/5 | Loss: 0.215 | Test Acc: 98.20%
  Epoch 2/5 | Loss: 0.064 | Test Acc: 98.95%
  Epoch 3/5 | Loss: 0.045 | Test Acc: 98.99%
  Epoch 4/5 | Loss: 0.036 | Test Acc: 98.95%
  Epoch 5/5 | Loss: 0.029 | Test Acc: 99.25%


'Store final CNN test accuracy.'

In [8]:
embs, labs = extract_embeddings(cnn, test_loader)
"""Extract CNN embeddings from the test set after CNN training."""
plot_tsne(embs, labs, 'CNN (128-dim)')
"""Plot CNN embedding space in 2D using t-SNE."""


  Running t-SNE: CNN (128-dim)...


'Plot CNN embedding space in 2D using t-SNE.'

---
## SOLUTION — Part 3: CNN + LSTM

In [9]:
class CNN_LSTM_MNIST(nn.Module):
    """
    CNN+LSTM architecture.

    The idea is to encode each image row with a small 1D CNN, then let an LSTM
    read the 28 row embeddings as a sequence from top to bottom.
    """
    def __init__(self, row_channels=144, hidden_lstm=250, classifier_hidden=96):
        """Create row encoder, LSTM sequence model, and classifier."""
        super().__init__()
        """Initialize nn.Module internals."""
        self.row_channels = row_channels
        """Dimension of each row embedding produced by the row CNN."""
        self.hidden_lstm = hidden_lstm
        """Dimension of the LSTM hidden state."""
        self.classifier_hidden = classifier_hidden
        """Intermediate classifier dimension used to match the CNN parameter budget."""
        """
        Parameter-matched setting:
        row_channels=144, hidden_lstm=250, classifier_hidden=96 gives exactly
        421,642 trainable parameters, matching the CNN baseline.
        """
        """
        Row encoder:
        - Each image row is treated as a 1D signal of length 28.
        - Conv1d learns local patterns along that row.
        - AdaptiveAvgPool1d(1) compresses each row to one vector.
        """
        self.row_cnn = nn.Sequential(
            nn.Conv1d(1, row_channels, kernel_size=3, padding=1), # (B*28,1,28) -> (B*28,row_channels,28)
            nn.ReLU(),                                            # non-linearity on row features
            nn.AdaptiveAvgPool1d(1)                               # (B*28,row_channels,28) -> (B*28,row_channels,1)
        )
        """Small CNN used independently on each row of the image."""
        self.lstm = nn.LSTM(input_size=row_channels, hidden_size=hidden_lstm, batch_first=True)
        """LSTM reads a sequence of 28 row embeddings."""
        self.fc = nn.Linear(hidden_lstm, classifier_hidden)
        """Project LSTM final state to a compact classifier embedding."""
        self.head = nn.Linear(classifier_hidden, 10)
        """Classifier embedding -> 10 digit logits."""

    def get_embeddings(self, x):
        """Return the final LSTM representation of the image."""
        B, C, H, W = x.shape
        """Read input shape: B images, 1 channel, 28 rows, 28 columns."""
        x = x.view(B*H, 1, W)
        """Treat every row as a separate 1D signal: (B,1,28,28) -> (B*28,1,28)."""
        row_features = self.row_cnn(x).squeeze(-1)
        """Encode each row and remove length dimension: (B*28,row_channels,1) -> (B*28,row_channels)."""
        row_features = row_features.view(B, H, self.row_channels)
        """Group rows back per image: (B*28,row_channels) -> (B,28,row_channels)."""
        _, (h_n, _) = self.lstm(row_features)
        """Run the LSTM and keep the final hidden state h_n."""
        return h_n.squeeze(0)

    def forward(self, x):
        """Return logits for the 10 digit classes."""
        emb = F.relu(self.fc(self.get_embeddings(x)))
        """Transform the LSTM embedding into the classifier hidden dimension."""
        return self.head(emb)


print('=== CNN + LSTM ===')
"""Print a section header for readability."""
cnn_lstm = CNN_LSTM_MNIST()
"""Create the CNN+LSTM model."""
print(f'Parameters: {count_params(cnn_lstm):,}')
"""Show parameter count; it should match the CNN baseline."""
histories['CNN+LSTM'] = train_model(cnn_lstm, train_loader)
"""Train CNN+LSTM and store accuracy after each epoch."""
results['CNN+LSTM'] = histories['CNN+LSTM'][-1]
"""Store final CNN+LSTM test accuracy."""


=== CNN + LSTM ===
Parameters: 421,642
  Epoch 1/5 | Loss: 0.780 | Test Acc: 87.59%
  Epoch 2/5 | Loss: 0.298 | Test Acc: 91.87%
  Epoch 3/5 | Loss: 0.230 | Test Acc: 93.24%
  Epoch 4/5 | Loss: 0.195 | Test Acc: 94.18%
  Epoch 5/5 | Loss: 0.171 | Test Acc: 94.67%


'Store final CNN+LSTM test accuracy.'

In [10]:
embs, labs = extract_embeddings(cnn_lstm, test_loader)
"""Extract CNN+LSTM embeddings from the test set after training."""
plot_tsne(embs, labs, 'CNN+LSTM (250-dim)')
"""Plot CNN+LSTM embedding space in 2D using t-SNE."""


  Running t-SNE: CNN+LSTM (250-dim)...


'Plot CNN+LSTM embedding space in 2D using t-SNE.'

---
## SOLUTION — Part 4: MiniViT

In [11]:
class SelfAttention(nn.Module):
    """
    Multi-head self-attention written from scratch.

    Input shape:
        x: (B, N, D)
        B = batch size
        N = number of tokens, here 49 image patches + 1 CLS token = 50
        D = embedding dimension, here 85 by default

    Goal:
        Let every token look at every other token and build a contextual representation.
    """
    def __init__(self, embed_dim, num_heads=5):
        """
        embed_dim is the full token dimension.
        num_heads splits the attention computation into parallel smaller attentions.
        With embed_dim=85 and num_heads=5, each head has dimension 17.
        """
        super().__init__()
        """Initialize the parent nn.Module class."""
        self.num_heads = num_heads
        """Store the number of attention heads."""
        self.head_dim  = embed_dim // num_heads
        """Each head receives only a slice of the embedding dimension."""
        self.scale     = self.head_dim ** -0.5
        """Scale factor 1/sqrt(head_dim) to stabilize the softmax."""
        self.WQ = nn.Linear(embed_dim, embed_dim)
        """WQ maps token embeddings to queries: what each token is looking for."""
        self.WK = nn.Linear(embed_dim, embed_dim)
        """WK maps token embeddings to keys: what each token advertises."""
        self.WV = nn.Linear(embed_dim, embed_dim)
        """WV maps token embeddings to values: information carried by each token."""
        self.Wo = nn.Linear(embed_dim, embed_dim)
        """Wo projects the concatenated heads back to embed_dim."""
        self.attn_weights = None
        """Store attention weights for later visualization of CLS attention maps."""

    def forward(self, x):
        """Compute self-attention for all tokens in the sequence."""
        B, N, D = x.shape
        """Read tensor dimensions: batch size, number of tokens, embedding size."""
        H, Hd = self.num_heads, self.head_dim
        """H is the number of heads, Hd is the dimension per head."""
        def rshp(t): return t.view(B, N, H, Hd).transpose(1,2)
        """Reshape from (B,N,D) to (B,H,N,Hd), so each head works separately."""
        Q, K, V = rshp(self.WQ(x)), rshp(self.WK(x)), rshp(self.WV(x))
        """Compute queries, keys, and values for all heads."""
        scores  = (Q @ K.transpose(-2,-1)) * self.scale
        """Compare every query with every key, then scale the scores."""
        weights = torch.softmax(scores, dim=-1)
        """Softmax turns scores into attention weights; each row sums to 1."""
        self.attn_weights = weights.detach()
        """Keep a detached copy for visualization without tracking gradients."""
        out = (weights @ V).transpose(1,2).contiguous().view(B, N, D)
        """Use attention weights to mix values, then concatenate heads back to (B,N,D)."""
        return self.Wo(out)

class TransformerBlock(nn.Module):
    """
    One reusable Transformer block.

    It contains:
    - self-attention to mix information between tokens,
    - LayerNorm to stabilize activations,
    - a feed-forward network to transform each token independently,
    - residual connections to make training easier.
    """
    def __init__(self, embed_dim, num_heads, mlp_dim=433):
        """Create the attention, normalization, and feed-forward sublayers."""
        super().__init__()
        self.attn = SelfAttention(embed_dim, num_heads)
        """Self-attention mixes information between all patch tokens and the CLS token."""
        self.ln1  = nn.LayerNorm(embed_dim)
        """LayerNorm before attention: this is the pre-norm Transformer style."""
        self.ln2  = nn.LayerNorm(embed_dim)
        """Second LayerNorm before the feed-forward network."""
        self.ffn  = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        """FFN expands D to mlp_dim, applies GELU, then returns to D."""
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        """Residual update: keep x and add the attention output."""
        x = x + self.ffn(self.ln2(x))
        """Residual update: keep x and add the feed-forward output."""
        return x

class PatchEmbedding(nn.Module):
    """
    Convert a 28x28 MNIST image into a sequence of patch tokens.

    With patch_size=4:
        28 / 4 = 7 patches per side
        7 * 7 = 49 image tokens
    """
    def __init__(self, img_size=28, patch_size=4, embed_dim=85):
        super().__init__()
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
        """
        Conv2d is used as a patch extractor.
        kernel_size=patch_size reads one patch.
        stride=patch_size moves from patch to patch without overlap.
        Output before flattening: (B, embed_dim, 7, 7).
        """
    def forward(self, x):
        x = self.proj(x)
        """Apply patch projection: (B,1,28,28) -> (B,embed_dim,7,7)."""
        x = x.flatten(2)
        """Flatten the 7x7 grid into 49 patch positions: (B,embed_dim,49)."""
        """Return tokens as (B,49,embed_dim), matching Transformer sequence format."""
        return x.transpose(1,2)

class MiniViT(nn.Module):
    """
    Parameter-matched MiniViT for MNIST classification.

    Defaults are calibrated to exactly 421,642 trainable parameters:
    embed_dim=85, num_heads=5, depth=4, mlp_dim=433.
    """
    def __init__(self, img_size=28, patch_size=4, embed_dim=85, num_heads=5, num_classes=10, depth=4, mlp_dim=433):
        """
        img_size: MNIST image side length, 28.
        patch_size: side length of each square patch, 4.
        embed_dim: vector dimension for each patch token.
        num_heads: number of attention heads inside each Transformer block.
        num_classes: 10 digit classes.
        depth: number of repeated Transformer blocks.
        mlp_dim: hidden dimension inside the feed-forward network.
        """
        super().__init__()
        n = (img_size // patch_size) ** 2
        """Number of image patch tokens: (28/4)^2 = 7^2 = 49."""
        self.patch_embed = PatchEmbedding(img_size, patch_size, embed_dim)
        """Module that converts the image into 49 patch embeddings."""
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        """Learnable CLS token: a summary token used for final classification."""
        self.pos_embed   = nn.Parameter(torch.randn(1, n+1, embed_dim) * 0.02)
        """Learnable positional embeddings for 49 patches plus 1 CLS token."""
        self.blocks      = nn.ModuleList([TransformerBlock(embed_dim, num_heads, mlp_dim) for _ in range(depth)])
        """Stack of Transformer blocks. Each block refines all token representations."""
        self.norm        = nn.LayerNorm(embed_dim)
        """Final normalization before classification."""
        self.head        = nn.Linear(embed_dim, num_classes)
        """Classification head: CLS embedding -> 10 digit logits."""
        self.drop        = nn.Dropout(0.1)
        """Dropout regularizes token embeddings during training. It has no trainable parameters."""
    def get_embeddings(self, x):
        B = x.size(0)
        """Read batch size."""
        x = self.patch_embed(x)
        """Convert image to patch tokens: (B,1,28,28) -> (B,49,D)."""
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        """Add CLS token at the beginning: (B,49,D) -> (B,50,D)."""
        x = self.drop(x + self.pos_embed)
        """Add positional embeddings so the model knows patch order."""
        for blk in self.blocks: x = blk(x)
        """Pass the token sequence through all Transformer blocks."""
        """Return only the CLS token embedding: (B,D)."""
        return self.norm(x[:, 0])
    def forward(self, x):
        """Return class logits by applying the classification head to the CLS embedding."""
        return self.head(self.get_embeddings(x))

print('=== MiniViT ===')
vit = MiniViT()
print(f'Parameters: {count_params(vit):,}')
histories['MiniViT'] = train_model(vit, train_loader, lr=3e-4)
results['MiniViT'] = histories['MiniViT'][-1]


=== MiniViT ===
Parameters: 421,642
  Epoch 1/5 | Loss: 1.185 | Test Acc: 81.27%
  Epoch 2/5 | Loss: 0.402 | Test Acc: 89.98%
  Epoch 3/5 | Loss: 0.243 | Test Acc: 94.81%
  Epoch 4/5 | Loss: 0.179 | Test Acc: 95.64%
  Epoch 5/5 | Loss: 0.145 | Test Acc: 96.29%


In [12]:
embs, labs = extract_embeddings(vit, test_loader)
"""Extract MiniViT CLS embeddings from the test set after training."""
plot_tsne(embs, labs, 'MiniViT (85-dim CLS)')
"""Plot MiniViT CLS embedding space in 2D using t-SNE."""


  Running t-SNE: MiniViT (85-dim CLS)...


'Plot MiniViT CLS embedding space in 2D using t-SNE.'

---
## SOLUTION — Part 5: Attention Map Visualization (Plotly)

In [13]:
def visualize_attention_maps(model, loader, n_digits=10):
    """
    Visualize which image patches the CLS token attends to.

    The first row shows original MNIST digits.
    The second row shows the attention map from the CLS token to image patches.
    """
    model.eval()
    """Use evaluation mode so Dropout is disabled."""
    samples = {}
    """Dictionary digit -> one example image."""
    for x, y in loader:
        """Loop over test batches until we collect one sample per digit."""
        for i in range(len(y)):
            """Inspect each image inside the batch."""
            d = y[i].item()
            """Read digit label as a Python integer."""
            if d not in samples:
                """Keep the first image found for each digit class."""
                samples[d] = x[i]
            if len(samples) == n_digits:
                """Stop once we have all requested digit classes."""
                break
        if len(samples) == n_digits:
            """Stop scanning batches once enough samples are collected."""
            break

    fig = make_subplots(
        rows=2,
        cols=10,
        subplot_titles=[f'Digit {d}' for d in range(10)] + ['Attention']*10,
        vertical_spacing=0.05,
        horizontal_spacing=0.01
    )
    """Create a 2-row grid: original images on top, attention maps below."""

    for digit in range(10):
        """Loop through digit classes 0 to 9."""
        img = samples[digit].unsqueeze(0).to(DEVICE)
        """Add batch dimension and move the image to the model device."""
        with torch.no_grad():
            """No gradients are needed for visualization."""
            _ = model(img)
            """Forward pass fills model.blocks[0].attn.attn_weights."""
        attn = model.blocks[0].attn.attn_weights
        """Attention shape should be (1, heads, 50, 50): CLS + 49 patches."""
        cls_attn = attn.mean(1)[0, 0, 1:].reshape(7, 7).cpu().numpy()
        """Average over heads, take CLS row, remove CLS-to-CLS, reshape 49 patches to 7x7."""
        attn_up = F.interpolate(
            torch.tensor(cls_attn).unsqueeze(0).unsqueeze(0).float(),
            size=(28, 28),
            mode='bilinear',
            align_corners=False
        ).squeeze().numpy()
        """Upsample the 7x7 patch attention map to 28x28 so it aligns with the image."""
        attn_up = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)
        """Normalize attention values to [0,1] for consistent colors."""
        orig = samples[digit][0].numpy()
        """Extract the original 28x28 image from shape (1,28,28)."""

        fig.add_trace(go.Heatmap(z=orig, colorscale='gray_r', showscale=False), row=1, col=digit+1)
        """Row 1: add the original digit image."""
        fig.add_trace(go.Heatmap(z=attn_up, colorscale='Hot', showscale=False), row=2, col=digit+1)
        """Row 2: add the CLS attention heatmap."""

    for k in fig.layout:
        """Loop over all Plotly layout entries to find axes."""
        if 'xaxis' in k or 'yaxis' in k:
            """Hide axis labels/ticks for both image and attention subplots."""
            fig.layout[k].update(showticklabels=False)
    fig.update_layout(
        title='MiniViT — CLS Attention Maps (head-averaged)',
        height=460,
        margin=dict(t=60)
    )
    """Set figure title, height, and top margin."""
    fig.show()
    """Render the attention visualization."""


visualize_attention_maps(vit, test_loader)
"""Run attention visualization after MiniViT training."""


'Run attention visualization after MiniViT training.'

---
## SOLUTION — Part 6: Comparison Dashboard (Plotly)

In [14]:
# --- Accuracy Curves ---
df_hist = pd.DataFrame(histories)
"""Convert history dictionary into a table: rows=epochs, columns=models."""
df_hist.index = range(1, len(df_hist)+1)
"""Use epoch numbers starting at 1 instead of 0."""
df_hist.index.name = 'Epoch'
"""Name the index so it becomes a clear column after reset_index()."""
df_long = df_hist.reset_index().melt(id_vars='Epoch', var_name='Architecture', value_name='Test Accuracy (%)')
"""Convert wide table to long format, which Plotly Express expects for colored lines."""

palette = {'MLP':'#4ECDC4','CNN':'#FF6B6B','CNN+LSTM':'#FFE66D','MiniViT':'#A29BFE'}
"""Use a fixed color per architecture so plots stay consistent."""
fig_acc = px.line(
    df_long,
    x='Epoch',
    y='Test Accuracy (%)',
    color='Architecture',
    markers=True,
    title='Architecture Comparison — MNIST Test Accuracy',
    color_discrete_map=palette,
    width=800,
    height=450
)
"""Plot test accuracy curves for all architectures on the same figure."""
fig_acc.update_traces(line=dict(width=2.5), marker=dict(size=8))
"""Make lines and markers easier to read in class."""
fig_acc.update_layout(yaxis_range=[90, 100])
"""Zoom y-axis to highlight differences near high accuracy."""
fig_acc.show()
"""Render the accuracy curve figure."""


'Render the accuracy curve figure.'

In [15]:
# --- Summary Bar Chart ---
models_dict = {'MLP': mlp, 'CNN': cnn, 'CNN+LSTM': cnn_lstm, 'MiniViT': vit}
"""Map architecture names to trained model objects so we can count parameters."""
biases = {
    'MLP':      'None',
    'CNN':      'Locality + Translation',
    'CNN+LSTM': 'Locality + Sequence',
    'MiniViT':  'Global Attention'
}
"""Short description of each architecture's inductive bias."""
summary = pd.DataFrame({
    'Architecture': list(results.keys()),
    'Test Accuracy (%)': [round(v, 2) for v in results.values()],
    'Parameters': [f'{count_params(m):,}' for m in models_dict.values()],
    'Inductive Bias': [biases[k] for k in results]
})
"""Build the final comparison table shown below the training experiments."""
print(summary.to_string(index=False))
"""Print the summary table without the pandas row index."""

fig_bar = px.bar(
    summary,
    x='Architecture',
    y='Test Accuracy (%)',
    color='Architecture',
    text='Test Accuracy (%)',
    color_discrete_map=palette,
    title='Final Test Accuracy by Architecture',
    width=700,
    height=420
)
"""Create a final bar chart of test accuracy by architecture."""
fig_bar.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
"""Show accuracy values above bars with two decimal places."""
fig_bar.update_layout(yaxis_range=[90, 100], showlegend=False)
"""Use the same zoomed y-axis and hide redundant legend."""
fig_bar.show()
"""Render the final comparison chart."""


Architecture  Test Accuracy (%) Parameters         Inductive Bias
         MLP              97.79    269,322                   None
         CNN              99.25    421,642 Locality + Translation
    CNN+LSTM              94.67    421,642    Locality + Sequence
     MiniViT              96.29    421,642       Global Attention


'Render the final comparison chart.'

---
## Reflection (Sample Answer)

**1. More inductive bias → better accuracy on MNIST?**  
Not necessarily. MNIST is so simple that even MLP achieves >97%. CNN adds spatial priors that give a real boost. CNN+LSTM adds sequential row-structure. MiniViT, without strong spatial priors, needs more data to learn from scratch — on MNIST it may slightly underperform CNN.

**2. Cleanest t-SNE?**  
CNN and CNN+LSTM typically show the cleanest class separation because their spatial/sequential inductive biases produce compact, highly discriminative features aligned with how digits are structured.

**3. Attention maps focus on ink?**  
Yes — CLS token attention concentrates on stroke pixels rather than white background, confirming the model learned to focus on discriminative regions without being told which pixels matter.

**4. CNN+LSTM beats MiniViT when?**  
On small datasets where sequential structure exists (ECG signals, handwriting stroke sequences, audio spectrograms by rows). The LSTM's sequential inductive bias prevents overfitting compared to a Transformer without sufficient data.

---
© 2026 Dr. Papa-Séga WADE - AIMS Senegal